# Advanced Problems with Solutions: Python Closures

This notebook contains 20+ advanced closure problems covering free variables, closure cells, `nonlocal`, shared extended scopes, multiple closure instances, late binding, nested closures, decorators, stateful functions, and practical closure design patterns.

Each problem includes a solution and explanatory notes.

## Problem 1: Inspecting Free Variables

Write a function `make_multiplier(n)` that returns a function multiplying its input by `n`.

Then inspect:

- `co_freevars`
- `__closure__`
- the actual captured value

In [1]:
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

times_7 = make_multiplier(7)

print(times_7(6))
print(times_7.__code__.co_freevars)
print(times_7.__closure__)
print(times_7.__closure__[0].cell_contents)

42
('n',)
(<cell at 0x00000203F32A0280: int object at 0x00007FFC88097468>,)
7


### Solution Explanation

`n` is not local to `multiplier`, but it is referenced inside it. Therefore, it is a free variable.

Python stores free variables in closure cells. The cell keeps the captured object alive even after `make_multiplier` has returned.

## Problem 2: Independent Closure Instances

Create two multipliers using the same factory:

- `times_2`
- `times_5`

Show that their closure cells are different.

In [2]:
times_2 = make_multiplier(2)
times_5 = make_multiplier(5)

print(times_2(10))
print(times_5(10))

cell_2 = times_2.__closure__[0]
cell_5 = times_5.__closure__[0]

print(cell_2 is cell_5)
print(cell_2.cell_contents)
print(cell_5.cell_contents)

20
50
False
2
5


### Solution Explanation

Each call to `make_multiplier` creates a fresh local scope. Therefore, `times_2` and `times_5` close over different cells.

## Problem 3: Stateful Counter with `nonlocal`

Implement `make_counter(start=0, step=1)`.

Each call should increment the counter by `step` and return the new value.

In [3]:
def make_counter(start=0, step=1):
    count = start

    def counter():
        nonlocal count
        count += step
        return count

    return counter

c = make_counter(10, 3)

print(c())
print(c())
print(c())

13
16
19


### Solution Explanation

`count += step` reassigns `count`, so `nonlocal count` is required.

`step` is only read, so it does not require `nonlocal`.

## Problem 4: Why `nonlocal` Is Required

Predict what happens when this code runs, then fix it.

In [4]:
def broken_counter():
    count = 0

    def inc():
        # count += 1 would fail without nonlocal
        nonlocal count
        count += 1
        return count

    return inc

c = broken_counter()
print(c())
print(c())

1
2


### Solution Explanation

Without `nonlocal`, Python treats `count` as local to `inc` because it is assigned inside `inc`.

Then `count += 1` tries to read a local variable before it has a value, causing `UnboundLocalError`.

## Problem 5: Shared Extended Scope

Write a function `make_bank_account(balance)` returning three closures:

- `deposit(amount)`
- `withdraw(amount)`
- `get_balance()`

All three closures should share the same balance cell.

In [5]:
def make_bank_account(balance):
    def deposit(amount):
        nonlocal balance
        if amount <= 0:
            raise ValueError('Deposit must be positive')
        balance += amount
        return balance

    def withdraw(amount):
        nonlocal balance
        if amount <= 0:
            raise ValueError('Withdrawal must be positive')
        if amount > balance:
            raise ValueError('Insufficient funds')
        balance -= amount
        return balance

    def get_balance():
        return balance

    return deposit, withdraw, get_balance

deposit, withdraw, get_balance = make_bank_account(100)

print(deposit(50))
print(withdraw(30))
print(get_balance())

print(deposit.__closure__[0] is withdraw.__closure__[0])
print(withdraw.__closure__[0] is get_balance.__closure__[0])

150
120
120
True
True


### Solution Explanation

All returned functions reference the same `balance` variable from the enclosing scope.

Therefore, they share the same closure cell.

## Problem 6: Avoiding Late Binding in Loops

Create five functions that multiply their input by `1`, `2`, `3`, `4`, and `5`.

First show the broken version, then fix it.

In [6]:
def broken_multipliers():
    funcs = []
    for n in range(1, 6):
        funcs.append(lambda x: x * n)
    return funcs

broken = broken_multipliers()
print([f(10) for f in broken])

def fixed_multipliers():
    funcs = []
    for n in range(1, 6):
        funcs.append(lambda x, factor=n: x * factor)
    return funcs

fixed = fixed_multipliers()
print([f(10) for f in fixed])

[50, 50, 50, 50, 50]
[10, 20, 30, 40, 50]


### Solution Explanation

The broken lambdas all share the same `n` cell.

By the time the functions are called, the loop has finished and `n == 5`.

The fixed version stores the current value of `n` in a default argument. Default arguments are evaluated when the function is created.

## Problem 7: Fix Late Binding Without Default Arguments

Fix the late-binding problem using a helper factory function instead of default arguments.

In [7]:
def make_one_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

def multipliers():
    return [make_one_multiplier(n) for n in range(1, 6)]

funcs = multipliers()
print([f(10) for f in funcs])

print(funcs[0].__closure__[0] is funcs[1].__closure__[0])

[10, 20, 30, 40, 50]
False


### Solution Explanation

Each call to `make_one_multiplier(n)` creates a new local scope and a new closure cell for `n`.

## Problem 8: Closure-Based Running Average

Implement `make_averager()`.

Each call should accept a new number and return the running average.

In [8]:
def make_averager():
    total = 0
    count = 0

    def add(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count

    return add

avg = make_averager()
print(avg(10))
print(avg(20))
print(avg(30))

10.0
15.0
20.0


### Solution Explanation

`total` and `count` persist because they are captured by the returned closure.

Both require `nonlocal` because both are reassigned.

## Problem 9: Closure-Based Memoization

Write a closure-based memoizer for a single-argument function.

In [9]:
def memoize(fn):
    cache = {}

    def inner(arg):
        if arg not in cache:
            cache[arg] = fn(arg)
        return cache[arg]

    return inner

def slow_square(n):
    print(f'Computing square of {n}')
    return n * n

square = memoize(slow_square)

print(square(5))
print(square(5))
print(square(6))
print(square(6))

print(square.__closure__[0].cell_contents)

Computing square of 5
25
25
Computing square of 6
36
36
{5: 25, 6: 36}


### Solution Explanation

`cache` is a mutable dictionary captured by the closure.

We mutate the dictionary, but we do not reassign `cache`, so `nonlocal cache` is not needed.

## Problem 10: Memoization with Multiple Arguments

Extend the previous memoizer to support `*args` and `**kwargs`.

In [10]:
def memoize_any(fn):
    cache = {}

    def inner(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    return inner

def combine(a, b, scale=1):
    print('Computing...')
    return (a + b) * scale

cached_combine = memoize_any(combine)

print(cached_combine(2, 3, scale=10))
print(cached_combine(2, 3, scale=10))
print(cached_combine(2, 3, scale=20))

Computing...
50
50
Computing...
100


### Solution Explanation

The cache key must be hashable.

`args` is already a tuple. `kwargs` is converted into a sorted tuple of key-value pairs so calls with the same keyword arguments produce the same cache key.

## Problem 11: Closure-Based Function Call Counter

Write a closure that wraps a function and tracks how many times it has been called.

In [11]:
def counted(fn):
    calls = 0

    def inner(*args, **kwargs):
        nonlocal calls
        calls += 1
        result = fn(*args, **kwargs)
        return result

    def get_calls():
        return calls

    inner.get_calls = get_calls
    return inner

@counted
def greet(name):
    return f'Hello, {name}!'

print(greet('Ada'))
print(greet('Grace'))
print(greet.get_calls())

Hello, Ada!
Hello, Grace!
2


### Solution Explanation

`calls` is shared between `inner` and `get_calls`.

The wrapper closure updates the value, while the accessor closure reads it.

## Problem 12: Preserving Function Metadata

Improve the previous decorator using `functools.wraps`.

In [12]:
from functools import wraps

def counted_preserve_metadata(fn):
    calls = 0

    @wraps(fn)
    def inner(*args, **kwargs):
        nonlocal calls
        calls += 1
        return fn(*args, **kwargs)

    def get_calls():
        return calls

    inner.get_calls = get_calls
    return inner

@counted_preserve_metadata
def add(a, b):
    'Add two numbers.'
    return a + b

print(add(2, 3))
print(add.__name__)
print(add.__doc__)
print(add.get_calls())

5
add
Add two numbers.
1


### Solution Explanation

`wraps(fn)` copies important metadata such as `__name__`, `__doc__`, and `__module__` from the wrapped function to the wrapper.

## Problem 13: Closure-Based Logger Factory

Create `make_logger(prefix)` that returns a logger function.

The logger should store all messages internally and expose a way to retrieve them.

In [13]:
def make_logger(prefix):
    messages = []

    def log(message):
        formatted = f'[{prefix}] {message}'
        messages.append(formatted)
        return formatted

    def history():
        return tuple(messages)

    log.history = history
    return log

error_log = make_logger('ERROR')

print(error_log('File not found'))
print(error_log('Invalid input'))
print(error_log.history())

[ERROR] File not found
[ERROR] Invalid input
('[ERROR] File not found', '[ERROR] Invalid input')


### Solution Explanation

`messages` is a captured list.

The closure mutates the list, so `nonlocal` is unnecessary.

Returning a tuple from `history` prevents outside code from directly mutating the internal list.

## Problem 14: Closure-Based Stack

Implement a stack using closures.

Return four functions:

- `push(value)`
- `pop()`
- `peek()`
- `size()`

In [14]:
def make_stack():
    items = []

    def push(value):
        items.append(value)

    def pop():
        if not items:
            raise IndexError('pop from empty stack')
        return items.pop()

    def peek():
        if not items:
            raise IndexError('peek from empty stack')
        return items[-1]

    def size():
        return len(items)

    return push, pop, peek, size

push, pop, peek, size = make_stack()

push('a')
push('b')
push('c')

print(peek())
print(size())
print(pop())
print(pop())
print(size())

c
3
c
b
1


### Solution Explanation

The list `items` lives in the enclosing function scope and remains alive because all returned closures reference it.

## Problem 15: Nested Closures

Write `make_power_offset(power)` returning a function `with_offset(offset)`.

`with_offset(offset)` should return a function that computes:

`x ** power + offset`

In [15]:
def make_power_offset(power):
    def with_offset(offset):
        def compute(x):
            return x ** power + offset
        return compute
    return with_offset

square_plus = make_power_offset(2)
square_plus_10 = square_plus(10)

print(square_plus_10(5))
print(square_plus.__code__.co_freevars)
print(square_plus_10.__code__.co_freevars)
print([cell.cell_contents for cell in square_plus_10.__closure__])

35
('power',)
('offset', 'power')
[10, 2]


### Solution Explanation

`compute` closes over both `power` and `offset`.

`power` comes from the outermost function. `offset` comes from the intermediate function.

## Problem 16: Closure with Resettable State

Create a counter closure that supports:

- `inc()`
- `reset()`
- `value()`

In [16]:
def make_resettable_counter(start=0):
    count = start

    def inc():
        nonlocal count
        count += 1
        return count

    def reset():
        nonlocal count
        count = start
        return count

    def value():
        return count

    return inc, reset, value

inc, reset, value = make_resettable_counter(100)

print(inc())
print(inc())
print(value())
print(reset())
print(value())

101
102
102
100
100


### Solution Explanation

`count` is reassigned in both `inc` and `reset`, so both require `nonlocal count`.

`start` is captured but never reassigned.

## Problem 17: Closure-Based Rate Limiter

Create `limit_calls(fn, max_calls)`.

The returned function should allow only `max_calls` successful calls. After that, it should raise `RuntimeError`.

In [17]:
def limit_calls(fn, max_calls):
    calls = 0

    def inner(*args, **kwargs):
        nonlocal calls
        if calls >= max_calls:
            raise RuntimeError('Call limit exceeded')
        calls += 1
        return fn(*args, **kwargs)

    return inner

def echo(value):
    return value

limited_echo = limit_calls(echo, 2)

print(limited_echo('first'))
print(limited_echo('second'))

try:
    print(limited_echo('third'))
except RuntimeError as ex:
    print(type(ex).__name__, ex)

first
second
RuntimeError Call limit exceeded


### Solution Explanation

The closure stores the number of successful calls in `calls`.

Because `calls` is reassigned, `nonlocal` is required.

## Problem 18: Closure-Based Validator Pipeline

Create a validation pipeline.

Each validator is a function that accepts a value and returns `True` or `False`.

`make_validator_pipeline(*validators)` should return a function that checks whether all validators pass.

In [18]:
def make_validator_pipeline(*validators):
    def validate(value):
        return all(validator(value) for validator in validators)
    return validate

is_int = lambda x: isinstance(x, int)
is_positive = lambda x: x > 0
is_even = lambda x: x % 2 == 0

positive_even_int = make_validator_pipeline(is_int, is_positive, is_even)

print(positive_even_int(10))
print(positive_even_int(-2))
print(positive_even_int(3))
print(positive_even_int('10'))

print(positive_even_int.__code__.co_freevars)

True
False
False
False
('validators',)


### Solution Explanation

`validators` is captured as a tuple.

The returned closure can use that tuple later even after `make_validator_pipeline` has finished executing.

## Problem 19: Closure-Based Configurable Formatter

Implement `make_formatter(prefix='', suffix='', transform=str)`.

The returned closure should format values using the captured configuration.

In [19]:
def make_formatter(prefix='', suffix='', transform=str):
    def formatter(value):
        return f'{prefix}{transform(value)}{suffix}'
    return formatter

money = make_formatter(prefix='$', transform=lambda x: f'{x:,.2f}')
brackets = make_formatter(prefix='[', suffix=']', transform=str.upper)

print(money(12345.678))
print(brackets('warning'))

$12,345.68
[WARNING]


### Solution Explanation

Closures are useful for freezing configuration into a callable.

`prefix`, `suffix`, and `transform` are all captured by the returned function.

## Problem 20: Closure vs Mutable Default Argument

Rewrite this function using a closure instead of a mutable default argument.

In [20]:
# Avoid this pattern:
def append_bad(value, items=[]):
    items.append(value)
    return items

print(append_bad(1))
print(append_bad(2))

# Better explicit closure-based version:
def make_appender():
    items = []

    def append(value):
        items.append(value)
        return list(items)

    return append

append_good = make_appender()

print(append_good(1))
print(append_good(2))

another_appender = make_appender()
print(another_appender('a'))

[1]
[1, 2]
[1]
[1, 2]
['a']


### Solution Explanation

Mutable default arguments persist across calls to the same function.

A closure makes the persistence explicit and allows each appender instance to have independent state.

## Problem 21: Manually Inspecting Closure Cell Contents

Write a helper function `describe_closure(fn)` that prints each free variable and its captured value.

In [21]:
def describe_closure(fn):
    freevars = fn.__code__.co_freevars
    closure = fn.__closure__

    if not freevars or not closure:
        print('No closure variables')
        return

    for name, cell in zip(freevars, closure):
        print(f'{name} -> {cell.cell_contents!r}')

def outer():
    x = [1, 2, 3]
    y = 'python'

    def inner():
        return x, y

    return inner

fn = outer()
describe_closure(fn)

x -> [1, 2, 3]
y -> 'python'


### Solution Explanation

`co_freevars` stores the names of free variables.

`__closure__` stores the corresponding cell objects.

Zipping them together gives names and values.

## Problem 22: Shared Cell Mutation Through One Closure

Create two closures that share a list. One closure adds values; the other reads the list.

In [22]:
def make_shared_list_tools():
    data = []

    def add(value):
        data.append(value)

    def get_all():
        return tuple(data)

    return add, get_all

add, get_all = make_shared_list_tools()

add('alpha')
add('beta')

print(get_all())
print(add.__closure__[0] is get_all.__closure__[0])

('alpha', 'beta')
True


### Solution Explanation

Both functions close over the same `data` cell.

Mutating the list through `add` is visible through `get_all`.

## Problem 23: Rebinding vs Mutating a Captured Object

Show the difference between mutating a captured list and rebinding the captured variable.

In [23]:
def make_tools():
    data = []

    def mutate(value):
        data.append(value)
        return data

    def rebind(value):
        nonlocal data
        data = [value]
        return data

    def view():
        return data

    return mutate, rebind, view

mutate, rebind, view = make_tools()

print(mutate(1))
print(mutate(2))
print(view())
print(rebind(99))
print(view())
print(mutate(100))
print(view())

[1]
[1, 2]
[1, 2]
[99]
[99]
[99, 100]
[99, 100]


### Solution Explanation

`data.append(value)` mutates the list object.

`data = [value]` reassigns the name `data`, so `nonlocal data` is required.

## Problem 24: Closure-Based Command Dispatcher

Create a command dispatcher using closures.

It should support:

- registering commands
- running commands
- listing commands

In [24]:
def make_dispatcher():
    commands = {}

    def register(name, fn):
        if name in commands:
            raise ValueError(f'Command {name!r} already registered')
        commands[name] = fn

    def run(name, *args, **kwargs):
        if name not in commands:
            raise KeyError(f'Unknown command: {name}')
        return commands[name](*args, **kwargs)

    def list_commands():
        return tuple(commands.keys())

    return register, run, list_commands

register, run, list_commands = make_dispatcher()

register('add', lambda a, b: a + b)
register('mul', lambda a, b: a * b)

print(list_commands())
print(run('add', 2, 3))
print(run('mul', 4, 5))

('add', 'mul')
5
20


### Solution Explanation

`commands` is a private dictionary stored in the closure.

Only the returned functions can access and modify it.

## Problem 25: Closure-Based Decorator Factory

Create a decorator factory `repeat(times)`.

The decorator should call the decorated function `times` times and return a list of results.

In [25]:
from functools import wraps

def repeat(times):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            return [fn(*args, **kwargs) for _ in range(times)]
        return inner
    return decorator

@repeat(3)
def say(word):
    return word.upper()

print(say('hello'))
print(say.__name__)
print(say.__code__.co_freevars)

['HELLO', 'HELLO', 'HELLO']
say
('fn', 'times')


### Solution Explanation

This has multiple closure layers:

- `repeat` captures `times`
- `decorator` captures `fn`
- `inner` uses both `times` and `fn`

## Final Challenge: Closure-Based Mini State Machine

Create a traffic light state machine using closures.

It should support:

- `current()`
- `next_state()`
- `reset()`

The states should cycle through:

`green -> yellow -> red -> green`

In [26]:
def make_traffic_light():
    states = ('green', 'yellow', 'red')
    index = 0

    def current():
        return states[index]

    def next_state():
        nonlocal index
        index = (index + 1) % len(states)
        return states[index]

    def reset():
        nonlocal index
        index = 0
        return states[index]

    return current, next_state, reset

current, next_state, reset = make_traffic_light()

print(current())
print(next_state())
print(next_state())
print(next_state())
print(reset())

green
yellow
red
green
green


### Final Challenge Explanation

`index` is captured and reassigned, so `next_state` and `reset` require `nonlocal index`.

`states` is captured but never reassigned.

This demonstrates how closures can model private state without creating a class.